# LeetCode 1929 — Concatenation of Array bằng Python

Link đề: https://leetcode.com/problems/concatenation-of-array/

Độ khó: **Easy** · Chủ đề: Array, Simulation

```python
class Solution:
    def getConcatenation(self, nums: list[int]) -> list[int]:

```

**Mục tiêu học:**

1. Đọc chính xác định nghĩa theo chỉ số: `ans[i] == nums[i]` và `ans[i + n] == nums[i]`.
2. Ba cách "thủ công": tạo mảng `2n` rồi gán theo chỉ số · `append` hai lượt · công thức **chỉ số vòng** `ans[j] = nums[j % n]`.
3. Ba cách Pythonic: toán tử ghép `nums + nums`, lặp `nums * 2`, `extend`.
4. Hiểu khác biệt **tạo list mới** vs **sửa list cũ**, và bẫy kinh điển: vừa duyệt vừa `append` vào chính list đó.
5. Biết `*` trên list là **sao chép nông** — vô hại với số nguyên, nhưng nguy hiểm với list lồng nhau.

## 1. Bài toán yêu cầu gì?

Cho `nums` độ dài `n`. Tạo `ans` độ dài `2n` sao cho với mọi `0 <= i < n`:

- `ans[i] == nums[i]` (nửa đầu là `nums`),
- `ans[i + n] == nums[i]` (nửa sau cũng là `nums`).

Nói cách khác, `ans` là `nums` **nối với chính nó**.

| nums | ans |
|---|---|
| `[1,2,1]` | `[1,2,1,1,2,1]` |
| `[1,3,2,1]` | `[1,3,2,1,1,3,2,1]` |
| `[7]` | `[7,7]` |

**Ràng buộc:** `1 <= n <= 1000`, `1 <= nums[i] <= 1000`.

In [1]:
examples = [
    ([1, 2, 1], [1, 2, 1, 1, 2, 1]),
    ([1, 3, 2, 1], [1, 3, 2, 1, 1, 3, 2, 1]),
    ([7], [7, 7]),
]
for nums, expected in examples:
    print(f"{nums} -> kỳ vọng {expected}")

[1, 2, 1] -> kỳ vọng [1, 2, 1, 1, 2, 1]
[1, 3, 2, 1] -> kỳ vọng [1, 3, 2, 1, 1, 3, 2, 1]
[7] -> kỳ vọng [7, 7]


## 2. Nền tảng Python cần biết

### 2.1. Tạo list độ dài cố định: `[0] * k`

`[0] * 6` tạo list 6 số 0. Hữu ích khi muốn **gán theo chỉ số** (`ans[j] = ...`) — gán vào chỉ số chưa tồn tại sẽ `IndexError`.

In [2]:
ans = [0] * 6
print(ans, len(ans))
ans[4] = 9
print(ans)
try:
    empty = []
    empty[0] = 1
except IndexError as e:
    print("Gán vào list rỗng -> IndexError:", e)

[0, 0, 0, 0, 0, 0] 6
[0, 0, 0, 0, 9, 0]
Gán vào list rỗng -> IndexError: list assignment index out of range


### 2.2. `append` (thêm một phần tử) và `extend` (thêm nhiều phần tử)

- `a.append(x)` thêm **một** phần tử `x` vào cuối.
- `a.extend(b)` thêm **từng phần tử** của `b` vào cuối.
- Cẩn thận: `a.append(b)` với `b` là list sẽ thêm **cả list** như một phần tử.

In [3]:
a = [1, 2]
a.append(3)
print("append 3   :", a)
a.extend([4, 5])
print("extend 4,5 :", a)
a.append([6, 7])
print("append list:", a, "<- list lồng!")

append 3   : [1, 2, 3]
extend 4,5 : [1, 2, 3, 4, 5]
append list: [1, 2, 3, 4, 5, [6, 7]] <- list lồng!


### 2.3. Toán tử `+` (ghép) và `*` (lặp) trên list

- `a + b` tạo list **mới** gồm `a` rồi `b`.
- `a * k` tạo list **mới** gồm `a` lặp `k` lần.

Cả hai **không** sửa `a`.

In [4]:
a = [1, 2, 1]
print(a + a)
print(a * 2)
print(a * 3)
print("a vẫn là:", a)

[1, 2, 1, 1, 2, 1]
[1, 2, 1, 1, 2, 1]
[1, 2, 1, 1, 2, 1, 1, 2, 1]
a vẫn là: [1, 2, 1]


### 2.4. Phép `%` cho chỉ số vòng (cyclic index)

Khi `j` chạy từ `0` tới `2n - 1`, `j % n` chạy `0, 1, ..., n-1, 0, 1, ..., n-1` — "quay vòng" về đầu.

In [5]:
n = 3
print([j % n for j in range(2 * n)])

[0, 1, 2, 0, 1, 2]


### 2.5. Tạo mới vs sửa tại chỗ — và `is`

`x is y` kiểm tra hai biến có trỏ tới **cùng một** object không. `a + a` tạo object mới; `a.extend(...)` sửa chính `a`.

In [6]:
a = [1, 2]
b_ = a + a
print("a + a là list mới?", b_ is not a)
a2 = a
a.extend([9])
print("a2 cũng thấy thay đổi:", a2, "| a2 is a:", a2 is a)

a + a là list mới? True
a2 cũng thấy thay đổi: [1, 2, 9] | a2 is a: True


## 3. Tính bằng tay theo định nghĩa chỉ số

`nums = [1,3,2,1]`, `n = 4`:

| i | nums[i] | gán `ans[i]` | gán `ans[i + 4]` |
|---:|---:|---|---|
| 0 | 1 | ans[0] = 1 | ans[4] = 1 |
| 1 | 3 | ans[1] = 3 | ans[5] = 3 |
| 2 | 2 | ans[2] = 2 | ans[6] = 2 |
| 3 | 1 | ans[3] = 1 | ans[7] = 1 |

→ `ans = [1,3,2,1,1,3,2,1]`.

Nhìn theo chiều ngược lại: với mỗi vị trí `j` của `ans`, giá trị là `nums[j % n]`.

In [7]:
nums = [1, 3, 2, 1]
n = len(nums)
for j in range(2 * n):
    print(f"ans[{j}] = nums[{j} % {n}] = nums[{j % n}] = {nums[j % n]}")

ans[0] = nums[0 % 4] = nums[0] = 1
ans[1] = nums[1 % 4] = nums[1] = 3
ans[2] = nums[2 % 4] = nums[2] = 2
ans[3] = nums[3 % 4] = nums[3] = 1
ans[4] = nums[4 % 4] = nums[0] = 1
ans[5] = nums[5 % 4] = nums[1] = 3
ans[6] = nums[6 % 4] = nums[2] = 2
ans[7] = nums[7 % 4] = nums[3] = 1


## 4. Cách 1 — Tạo mảng `2n`, gán theo đúng định nghĩa (hint của đề)

Đây là cách "dịch thẳng" đề bài sang code, chạy được ở mọi ngôn ngữ.

In [8]:
def get_concatenation_index(nums):
    n = len(nums)
    ans = [0] * (2 * n)
    for i in range(n):
        ans[i] = nums[i]
        ans[i + n] = nums[i]
    return ans

for nums, expected in examples:
    print(get_concatenation_index(nums), "| kỳ vọng", expected)

[1, 2, 1, 1, 2, 1] | kỳ vọng [1, 2, 1, 1, 2, 1]
[1, 3, 2, 1, 1, 3, 2, 1] | kỳ vọng [1, 3, 2, 1, 1, 3, 2, 1]
[7, 7] | kỳ vọng [7, 7]


### Code nộp LeetCode — Cách 1

In [9]:
class Solution:
    def getConcatenation(self, nums: list[int]) -> list[int]:
        n = len(nums)
        ans = [0] * (2 * n)
        for i in range(n):
            ans[i] = nums[i]
            ans[i + n] = nums[i]
        return ans


print(Solution().getConcatenation([1, 2, 1]))

[1, 2, 1, 1, 2, 1]


**Độ phức tạp:** thời gian `O(n)` (một vòng `n` bước, mỗi bước 2 phép gán), bộ nhớ `O(n)` cho `ans` — đây là kích thước **đầu ra bắt buộc**, không thể ít hơn.

## 5. Cách 2 — `append` hai lượt

Không cần biết trước độ dài: duyệt `nums` hai lần, thêm từng phần tử.

In [10]:
def get_concatenation_append(nums):
    ans = []
    for _ in range(2):          # hai lượt
        for x in nums:
            ans.append(x)
    return ans

for nums, expected in examples:
    print(get_concatenation_append(nums), "| kỳ vọng", expected)

[1, 2, 1, 1, 2, 1] | kỳ vọng [1, 2, 1, 1, 2, 1]
[1, 3, 2, 1, 1, 3, 2, 1] | kỳ vọng [1, 3, 2, 1, 1, 3, 2, 1]
[7, 7] | kỳ vọng [7, 7]


`for _ in range(2)`: dấu gạch dưới `_` là quy ước cho "biến không dùng tới" — ta chỉ cần lặp 2 lần.

## 6. Cách 3 — Chỉ số vòng `nums[j % n]`

Một list comprehension duy nhất, nhìn từ phía **đầu ra**: mỗi `ans[j]` lấy từ `nums[j % n]`. Ý tưởng này khái quát ngay sang "lặp `k` lần" hay các bài mảng vòng (circular array).

In [11]:
def get_concatenation_mod(nums):
    n = len(nums)
    return [nums[j % n] for j in range(2 * n)]

for nums, expected in examples:
    print(get_concatenation_mod(nums), "| kỳ vọng", expected)

[1, 2, 1, 1, 2, 1] | kỳ vọng [1, 2, 1, 1, 2, 1]
[1, 3, 2, 1, 1, 3, 2, 1] | kỳ vọng [1, 3, 2, 1, 1, 3, 2, 1]
[7, 7] | kỳ vọng [7, 7]


## 7. Lời giải chính — Pythonic: `nums + nums` (hoặc `nums * 2`)

Python có sẵn phép **ghép list**. Định nghĩa của đề ("`ans` là `nums` nối với `nums`") chính là `nums + nums`.

- `nums + nums` — đọc là "ghép".
- `nums * 2` — đọc là "lặp 2 lần".

Cả hai tạo list mới, không sửa `nums`, chạy bằng code C bên trong nên rất nhanh.

In [12]:
def get_concatenation_plus(nums):
    return nums + nums


def get_concatenation_times(nums):
    return nums * 2


for nums, expected in examples:
    print(get_concatenation_plus(nums), get_concatenation_times(nums), "| kỳ vọng", expected)

[1, 2, 1, 1, 2, 1] [1, 2, 1, 1, 2, 1] | kỳ vọng [1, 2, 1, 1, 2, 1]
[1, 3, 2, 1, 1, 3, 2, 1] [1, 3, 2, 1, 1, 3, 2, 1] | kỳ vọng [1, 3, 2, 1, 1, 3, 2, 1]
[7, 7] [7, 7] | kỳ vọng [7, 7]


### Cách 6 — `extend` (sửa tại chỗ) và bản sao an toàn

`ans = nums[:]` (hoặc `list(nums)`) tạo bản sao, rồi `ans.extend(nums)`. Nếu gọi `nums.extend(nums)` trực tiếp thì cũng ra đúng kết quả nhưng **sửa input** của người gọi.

In [13]:
def get_concatenation_extend(nums):
    ans = nums[:]           # bản sao
    ans.extend(nums)
    return ans

data = [1, 2, 1]
print(get_concatenation_extend(data), "| input vẫn là", data)

data2 = [1, 2, 1]
data2.extend(data2)         # Python xử lý đúng khi extend chính nó
print("extend chính nó:", data2, "| nhưng input đã bị đổi!")

[1, 2, 1, 1, 2, 1] | input vẫn là [1, 2, 1]
extend chính nó: [1, 2, 1, 1, 2, 1] | nhưng input đã bị đổi!


## 8. TRACE từng bước

Trace Cách 1 với `nums = [1,2,1]`: in trạng thái `ans` sau mỗi bước để thấy hai nửa được "lấp" song song.

In [14]:
def trace_index(nums):
    n = len(nums)
    ans = [None] * (2 * n)            # None để dễ thấy ô chưa gán
    print(f"n = {n}, ans ban đầu = {ans}")
    for i in range(n):
        ans[i] = nums[i]
        ans[i + n] = nums[i]
        print(f"  i={i}: ans[{i}] = ans[{i + n}] = nums[{i}] = {nums[i]} -> {ans}")
    return ans

trace_index([1, 2, 1])

n = 3, ans ban đầu = [None, None, None, None, None, None]
  i=0: ans[0] = ans[3] = nums[0] = 1 -> [1, None, None, 1, None, None]
  i=1: ans[1] = ans[4] = nums[1] = 2 -> [1, 2, None, 1, 2, None]
  i=2: ans[2] = ans[5] = nums[2] = 1 -> [1, 2, 1, 1, 2, 1]


[1, 2, 1, 1, 2, 1]

### Trace Cách 2 (`append`) — `ans` lớn dần

In [15]:
def trace_append(nums):
    ans = []
    for lap in range(2):
        for i, x in enumerate(nums):
            ans.append(x)
            print(f"  lượt {lap + 1}, i={i}: append {x} -> {ans}")
    return ans

trace_append([1, 3, 2, 1])

  lượt 1, i=0: append 1 -> [1]
  lượt 1, i=1: append 3 -> [1, 3]
  lượt 1, i=2: append 2 -> [1, 3, 2]
  lượt 1, i=3: append 1 -> [1, 3, 2, 1]
  lượt 2, i=0: append 1 -> [1, 3, 2, 1, 1]
  lượt 2, i=1: append 3 -> [1, 3, 2, 1, 1, 3]
  lượt 2, i=2: append 2 -> [1, 3, 2, 1, 1, 3, 2]
  lượt 2, i=3: append 1 -> [1, 3, 2, 1, 1, 3, 2, 1]


[1, 3, 2, 1, 1, 3, 2, 1]

## 9. Code nộp LeetCode — lời giải chính

In [16]:
class Solution:
    def getConcatenation(self, nums: list[int]) -> list[int]:
        return nums + nums


sol = Solution()
for nums, expected in examples:
    assert sol.getConcatenation(nums) == expected
print("Solution (nums + nums) PASS")

Solution (nums + nums) PASS


## 10. Vì sao đúng?

**Cách 1:** sau vòng lặp, với mỗi `i ∈ [0, n)` ta đã gán đúng `ans[i] = nums[i]` và `ans[i+n] = nums[i]`. Tập chỉ số `{i} ∪ {i + n}` với `i ∈ [0, n)` phủ **đúng một lần** mọi chỉ số `0..2n-1` → không ô nào bị bỏ sót hay gán hai lần.

**Cách 3:** với `j < n`, `j % n = j` → `ans[j] = nums[j]`. Với `n <= j < 2n`, viết `j = i + n` thì `j % n = i` → `ans[i + n] = nums[i]`. Đúng định nghĩa.

**`nums + nums`:** theo định nghĩa của phép ghép list trong Python, `(a + b)[k] = a[k]` nếu `k < len(a)`, và `b[k - len(a)]` nếu không. Với `a = b = nums` ta được đúng hai điều kiện của đề.

## 11. Độ phức tạp và bảng so sánh

`n = len(nums)`. Mọi cách đều `O(n)` thời gian và `O(n)` bộ nhớ cho đầu ra — không thể tốt hơn vì phải ghi `2n` phần tử.

| Cách | Code | Sửa input? | Ghi chú |
|---|---|:---:|---|
| 1. Gán theo chỉ số | `ans[i] = ans[i+n] = nums[i]` | Không | bám sát định nghĩa, mọi ngôn ngữ |
| 2. `append` hai lượt | `for _ in range(2): for x in nums: ...` | Không | không cần biết trước độ dài |
| 3. Chỉ số vòng | `[nums[j % n] for j in range(2n)]` | Không | khái quát cho mảng vòng / lặp `k` lần |
| 4. Ghép | `nums + nums` | Không | ngắn, rõ nhất |
| 5. Lặp | `nums * 2` | Không | ngắn; nhớ: sao chép **nông** |
| 6. `extend` | `ans = nums[:]; ans.extend(nums)` | Không (có bản sao) | `nums.extend(nums)` thì **có** sửa |

Đo thời gian thực tế cho thấy các cách dựng sẵn (`+`, `*`) nhanh hơn vòng lặp Python nhiều lần (cell dưới).

In [17]:
import time

big = list(range(1, 1001)) * 100          # 100.000 phần tử
solvers = [get_concatenation_index, get_concatenation_append, get_concatenation_mod,
           get_concatenation_plus, get_concatenation_times, get_concatenation_extend]
ref = big + big
for f in solvers:
    t0 = time.perf_counter()
    for _ in range(5):
        out = f(big)
    dt = (time.perf_counter() - t0) / 5
    assert out == ref
    print(f"{f.__name__:28} {dt * 1000:8.3f} ms")

get_concatenation_index         5.647 ms
get_concatenation_append        3.123 ms
get_concatenation_mod           7.281 ms
get_concatenation_plus          0.404 ms
get_concatenation_times         0.300 ms
get_concatenation_extend        0.380 ms


(Số đo phụ thuộc máy; điều quan trọng là **thứ tự** nhanh/chậm, không phải con số chính xác.)

## 12. Bộ test kiểm tra

In [18]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:28} PASS")

tests = [
    ("ví dụ 1", [1, 2, 1], [1, 2, 1, 1, 2, 1]),
    ("ví dụ 2", [1, 3, 2, 1], [1, 3, 2, 1, 1, 3, 2, 1]),
    ("n = 1", [7], [7, 7]),
    ("giá trị biên 1000", [1000, 1], [1000, 1, 1000, 1]),
    ("toàn trùng", [5, 5, 5], [5, 5, 5, 5, 5, 5]),
]
for name, nums, e in tests:
    original = nums[:]
    for f in solvers:
        assert f(nums) == e, (name, f.__name__)
        assert nums == original, ("input bị sửa!", f.__name__)
    check(name + " (6 cách)", get_concatenation_plus(nums), e)

ví dụ 1 (6 cách)             PASS
ví dụ 2 (6 cách)             PASS
n = 1 (6 cách)               PASS
giá trị biên 1000 (6 cách)   PASS
toàn trùng (6 cách)          PASS


In [19]:
import random

random.seed(1929)
for _ in range(1000):
    n_ = random.randint(1, 50)
    nums = [random.randint(1, 1000) for _ in range(n_)]
    ans = get_concatenation_index(nums)
    # kiểm tra trực tiếp định nghĩa của đề
    assert len(ans) == 2 * n_
    assert all(ans[i] == nums[i] and ans[i + n_] == nums[i] for i in range(n_))
    assert all(f(nums) == ans for f in solvers)
print("1000 mảng ngẫu nhiên: mọi cách thỏa định nghĩa ans[i] == ans[i+n] == nums[i]")

1000 mảng ngẫu nhiên: mọi cách thỏa định nghĩa ans[i] == ans[i+n] == nums[i]


Test ngẫu nhiên ở trên kiểm tra trực tiếp **định nghĩa của đề** (không chỉ so sánh các cách với nhau), nên nếu tất cả cùng sai một kiểu thì vẫn bị phát hiện.

## 13. Lỗi người mới hay gặp

### Lỗi 1: Vừa duyệt vừa `append` vào chính list đó → vòng lặp vô hạn

```python
for x in nums:
    nums.append(x)     # SAI: nums dài ra mãi, vòng for không bao giờ hết
```

Cell dưới mô phỏng có **giới hạn an toàn** để không treo notebook.

In [20]:
nums = [1, 2, 1]
steps = 0
for x in nums:
    nums.append(x)
    steps += 1
    if steps >= 10:            # chốt an toàn cho demo
        print("Đã lặp 10 lần mà chưa dừng! len(nums) =", len(nums))
        break

Đã lặp 10 lần mà chưa dừng! len(nums) = 13


Sửa: duyệt một **bản sao** (`for x in nums[:]`) hoặc dùng `nums + nums` / `extend`.

### Lỗi 2: `append(nums)` thay vì `extend(nums)`

In [21]:
nums = [1, 2, 1]
ans = nums[:]
ans.append(nums)
print("Sai :", ans)
ans = nums[:]
ans.extend(nums)
print("Đúng:", ans)

Sai : [1, 2, 1, [1, 2, 1]]
Đúng: [1, 2, 1, 1, 2, 1]


### Lỗi 3: `ans = nums` rồi `extend` — tưởng là bản sao

```python
ans = nums          # KHÔNG sao chép
ans.extend(nums)    # sửa luôn nums của người gọi
```

In [22]:
nums = [1, 2, 1]
ans = nums
ans.extend(nums)
print("ans =", ans, "| nums cũng bị đổi:", nums, "| ans is nums:", ans is nums)

ans = [1, 2, 1, 1, 2, 1] | nums cũng bị đổi: [1, 2, 1, 1, 2, 1] | ans is nums: True


### Lỗi 4: Gán `ans[i + n]` khi `ans` chưa đủ dài

```python
ans = nums[:]           # độ dài n
ans[i + n] = nums[i]    # SAI: IndexError
```

Phải tạo `[0] * (2 * n)` trước, hoặc dùng `append`.

### Lỗi 5: `* 2` với list lồng nhau — sao chép **nông**

Với bài này (list số nguyên) không sao. Nhưng nếu phần tử là list, `x * 2` lặp lại **cùng** các list con.

In [23]:
grid = [[1], [2]]
doubled = grid * 2
doubled[0].append(99)
print(doubled, "<- doubled[2] cũng đổi vì là cùng object với doubled[0]")
print("doubled[0] is doubled[2]:", doubled[0] is doubled[2])

[[1, 99], [2], [1, 99], [2]] <- doubled[2] cũng đổi vì là cùng object với doubled[0]
doubled[0] is doubled[2]: True


## 14. Mẹo nhận dạng pattern và biến thể

- "Ghép / lặp mảng" → `+`, `*`, `extend`, hoặc chỉ số vòng `j % n`.
- **Mảng vòng (circular array)**: thay vì thật sự nhân đôi mảng, dùng `nums[j % n]` với `j` chạy tới `2n` — tiết kiệm bộ nhớ. Đây là mẹo chuẩn của LeetCode 503 (Next Greater Element II), 918 (Maximum Sum Circular Subarray), 2134.

Bài liên quan: 1920 (Build Array from Permutation), 1470 (Shuffle the Array), 503, 189 (Rotate Array).

### Biến thể 1: lặp `k` lần

In [24]:
def repeat_k(nums, k):
    n = len(nums)
    return [nums[j % n] for j in range(k * n)]

print(repeat_k([1, 2], 3), "==", [1, 2] * 3)

[1, 2, 1, 2, 1, 2] == [1, 2, 1, 2, 1, 2]


### Biến thể 2: dùng chỉ số vòng thay vì nhân đôi — "phần tử lớn hơn kế tiếp" trên mảng vòng (503)

Mỗi phần tử tìm số lớn hơn đầu tiên khi đi **theo vòng**. Thay vì tạo `nums + nums`, duyệt `j` từ `2n-1` về 0 với `nums[j % n]` và một **stack**.

In [25]:
def next_greater_circular(nums):
    n = len(nums)
    res = [-1] * n
    stack = []                      # chứa các giá trị "ứng viên", giảm dần từ đáy lên đỉnh
    for j in range(2 * n - 1, -1, -1):
        x = nums[j % n]
        while stack and stack[-1] <= x:
            stack.pop()
        if j < n:
            res[j] = stack[-1] if stack else -1
        stack.append(x)
    return res

print(next_greater_circular([1, 2, 1]))     # [2, -1, 2]
print(next_greater_circular([1, 2, 3, 4, 3]))

[2, -1, 2]
[2, 3, 4, -1, 4]


## 15. Bài tập tự luyện (TODO)

### Bài 1 — Dự đoán output

Không chạy code, ghi kết quả của 4 biểu thức sau vào `ex1_answers`: `[0] * 3`, `[1, 2] + [3]`, `[[0]] * 2`, `len([4, 5] * 4)`.

In [26]:
ex1_answers = None   # TODO: list 4 giá trị

### Bài 2 — Điền chỗ trống: gán theo chỉ số

In [27]:
def ex2_get_concatenation(nums):
    n = len(nums)
    ans = [0] * (2 * n)
    for i in range(n):
        # TODO: gán hai ô
        pass
    return ans

# for nums, e in examples: assert ex2_get_concatenation(nums) == e

### Bài 3 — Ghép hai mảng xen kẽ

Viết `interleave(a, b)` với `len(a) == len(b)`: trả về `[a0, b0, a1, b1, ...]` (giống LeetCode 1470).

In [28]:
def interleave(a, b):
    # TODO
    pass

# assert interleave([2, 5, 1], [3, 4, 7]) == [2, 3, 5, 4, 1, 7]

### Bài 4 — Lặp `k` lần không dùng `*`

Viết `repeat_without_star(nums, k)` chỉ dùng vòng lặp và `append` (hoặc chỉ số vòng).

In [29]:
def repeat_without_star(nums, k):
    # TODO
    pass

# assert repeat_without_star([1, 2], 3) == [1, 2, 1, 2, 1, 2]
# assert repeat_without_star([5], 0) == []

### Bài 5 — Xoay mảng phải `k` bước bằng chỉ số vòng (LeetCode 189, bản tạo mảng mới)

Ví dụ `[1,2,3,4,5,6,7]`, `k = 3` → `[5,6,7,1,2,3,4]`.

In [30]:
def rotate_right(nums, k):
    # TODO: ans[(i + k) % n] = nums[i]
    pass

# assert rotate_right([1, 2, 3, 4, 5, 6, 7], 3) == [5, 6, 7, 1, 2, 3, 4]
# assert rotate_right([1, 2], 5) == [2, 1]

### Bài 6 — Mini challenge: tổng đoạn con lớn nhất trên `nums + nums` với độ dài tối đa `n`

Cho `nums`, xét mảng vòng (tức `nums + nums`), tìm tổng lớn nhất của một đoạn con **liên tiếp** có độ dài từ 1 đến `n`. Brute force `O(n²)` là đủ (gợi ý: chỉ số vòng `% n`, không cần tạo mảng nhân đôi).

In [31]:
def max_circular_subarray_bruteforce(nums):
    # TODO
    pass

# assert max_circular_subarray_bruteforce([5, -3, 5]) == 10
# assert max_circular_subarray_bruteforce([-3, -2, -3]) == -2
# assert max_circular_subarray_bruteforce([1, -2, 3, -2]) == 3

## 16. Gợi ý 3 mức

**Bài 1.** (1) `*` lặp, `+` ghép. (2) `[[0]] * 2` có 2 phần tử, mỗi phần tử là `[0]`. (3) `len` = 2 × 4.

**Bài 2.** (1) Mỗi `i` gán 2 ô. (2) Ô `i` và ô `i + n`. (3) `ans[i] = ans[i + n] = nums[i]`.

**Bài 3.** (1) Duyệt `i` từ 0 tới `n-1`. (2) Mỗi bước thêm 2 phần tử. (3) `ans.append(a[i]); ans.append(b[i])`.

**Bài 4.** (1) `k` lượt. (2) Mỗi lượt `extend` hoặc `append` từng phần tử. (3) Hoặc `[nums[j % n] for j in range(k * n)]`.

**Bài 5.** (1) Phần tử ở `i` đi tới `(i + k) % n`. (2) Tạo `ans = [0] * n`. (3) `k` lớn hơn `n` vẫn đúng nhờ `%`.

**Bài 6.** (1) Chọn điểm bắt đầu `s ∈ [0, n)`. (2) Cộng dần `nums[(s + L) % n]` với `L = 0..n-1`. (3) Cập nhật `best` sau mỗi lần cộng.

## 17. Đáp án tham khảo (chỉ xem sau khi tự làm)

In [32]:
ans_ex1 = [[0, 0, 0], [1, 2, 3], [[0], [0]], 8]


def ans_ex2_get_concatenation(nums):
    n = len(nums)
    ans = [0] * (2 * n)
    for i in range(n):
        ans[i] = ans[i + n] = nums[i]
    return ans


def ans_interleave(a, b):
    ans = []
    for i in range(len(a)):
        ans.append(a[i])
        ans.append(b[i])
    return ans


def ans_repeat_without_star(nums, k):
    ans = []
    for _ in range(k):
        for x in nums:
            ans.append(x)
    return ans


def ans_rotate_right(nums, k):
    n = len(nums)
    ans = [0] * n
    for i in range(n):
        ans[(i + k) % n] = nums[i]
    return ans


def ans_max_circular_subarray_bruteforce(nums):
    n = len(nums)
    best = float("-inf")
    for s in range(n):
        total = 0
        for L in range(n):
            total += nums[(s + L) % n]
            best = max(best, total)
    return best


assert ans_ex1 == [[0] * 3, [1, 2] + [3], [[0]] * 2, len([4, 5] * 4)]
for nums, e in examples:
    assert ans_ex2_get_concatenation(nums) == e
assert ans_interleave([2, 5, 1], [3, 4, 7]) == [2, 3, 5, 4, 1, 7]
assert ans_repeat_without_star([1, 2], 3) == [1, 2, 1, 2, 1, 2] and ans_repeat_without_star([5], 0) == []
assert ans_rotate_right([1, 2, 3, 4, 5, 6, 7], 3) == [5, 6, 7, 1, 2, 3, 4]
assert ans_rotate_right([1, 2], 5) == [2, 1]
assert ans_max_circular_subarray_bruteforce([5, -3, 5]) == 10
assert ans_max_circular_subarray_bruteforce([-3, -2, -3]) == -2
assert ans_max_circular_subarray_bruteforce([1, -2, 3, -2]) == 3

# đối chiếu bài 6 với cách tạo thật nums + nums
random.seed(6)
for _ in range(300):
    arr = [random.randint(-5, 5) for _ in range(random.randint(1, 7))]
    dbl = arr + arr
    n_ = len(arr)
    expect = max(sum(dbl[s:s + L]) for s in range(n_) for L in range(1, n_ + 1))
    assert ans_max_circular_subarray_bruteforce(arr) == expect
print("Tất cả đáp án PASS!")

Tất cả đáp án PASS!


## 18. Template nhớ nhanh + checklist submit

```python
class Solution:
    def getConcatenation(self, nums: list[int]) -> list[int]:
        return nums + nums            # hoặc nums * 2

# Bản "mọi ngôn ngữ":
#   ans = [0] * (2 * n)
#   for i in range(n): ans[i] = ans[i + n] = nums[i]
# Bản chỉ số vòng:
#   [nums[j % n] for j in range(2 * n)]
```

**Khung tư duy:** đọc định nghĩa theo chỉ số → có thể "dịch thẳng" (gán `ans[i]`, `ans[i+n]`) hoặc nhìn từ đầu ra (`ans[j] = nums[j % n]`) → dùng công cụ dựng sẵn khi ngôn ngữ cho phép.

**Checklist trước khi submit:**

- [ ] Độ dài kết quả đúng `2n`?
- [ ] Không vừa duyệt vừa `append` vào chính `nums`?
- [ ] `extend` chứ không phải `append(list)`?
- [ ] Không vô tình sửa `nums` (`ans = nums` không phải bản sao)?
- [ ] Tên method `getConcatenation`, trả `list[int]`?